In [1]:
import pandas as pd
import numpy as np

customers = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/dataset/customers_cleaned.parquet"
)

articles = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/dataset/articles_cleaned.parquet"
)

transactions = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/dataset/transactions_cleaned.parquet"
)

print("Datasets loaded!")

Datasets loaded!


In [2]:
print(customers.shape)
print(articles.shape)
print(transactions.shape)

(1371980, 7)
(105542, 25)
(28813419, 5)


Extract purchase behavior from date.

In [3]:
transactions['year'] = (
    transactions['t_dat']
    .dt.year
)

transactions['month'] = (
    transactions['t_dat']
    .dt.month
)

transactions['day'] = (
    transactions['t_dat']
    .dt.day
)

transactions['day_of_week'] = (
    transactions['t_dat']
    .dt.day_name()
)

transactions['week_of_year'] = (
    transactions['t_dat']
    .dt.isocalendar()
    .week
)

transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id,year,month,day,day_of_week,week_of_year
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2,2018,9,20,Thursday,38
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2,2018,9,20,Thursday,38
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2,2018,9,20,Thursday,38
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2,2018,9,20,Thursday,38
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2,2018,9,20,Thursday,38


How active a customer is.

In [4]:
customer_frequency = (
    transactions
    .groupby('customer_id')
    .size()
    .reset_index(
        name='purchase_frequency'
    )
)

customer_frequency.head()

,customer_id,purchase_frequency
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,19
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,78
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,15
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,2
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,13


How recently user purchased.

In [5]:
latest_date = (
    transactions['t_dat']
    .max()
)

customer_recency = (
    transactions
    .groupby('customer_id')
    ['t_dat']
    .max()
    .reset_index()
)

customer_recency[
    'days_since_last_purchase'
] = (
    latest_date -
    customer_recency['t_dat']
).dt.days

customer_recency.head()

,customer_id,t_dat,days_since_last_purchase
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,2020-09-05,17
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,2020-07-08,76
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2020-09-15,7
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,2019-06-09,471
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,2020-08-12,41


Customer average spend

In [6]:
customer_avg_spend = (
    transactions
    .groupby('customer_id')
    ['price']
    .mean()
    .reset_index(
        name='avg_spend'
    )
)

customer_avg_spend.head()

,customer_id,avg_spend
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.028628
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.030926
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.040435
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.030492
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,0.036130


Product purchase counts.

In [7]:
product_popularity = (
    transactions
    .groupby('article_id')
    .size()
    .reset_index(
        name='purchase_count'
    )
)

product_popularity.head()

,article_id,purchase_count
0,108775015,7535
1,108775044,5666
2,108775051,177
3,110065001,982
4,110065002,482


In [8]:
monthly_popularity = (
    transactions
    .groupby(
        ['month', 'article_id']
    )
    .size()
    .reset_index(
        name='monthly_purchase_count'
    )
)

monthly_popularity.head()

,month,article_id,monthly_purchase_count
0,1,108775015,990
1,1,108775044,507
2,1,110065001,89
3,1,110065002,113
4,1,110065011,165


Create customer feature table.

In [9]:
customer_features = (
    customers
    .merge(
        customer_frequency,
        on='customer_id',
        how='left'
    )
    .merge(
        customer_recency[
            [
                'customer_id',
                'days_since_last_purchase'
            ]
        ],
        on='customer_id',
        how='left'
    )
    .merge(
        customer_avg_spend,
        on='customer_id',
        how='left'
    )
)

customer_features.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,purchase_frequency,days_since_last_purchase,avg_spend
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,0.0,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,19.0,17.0,0.028628
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.0,0.0,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,78.0,76.0,0.030926
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,0.0,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,15.0,7.0,0.040435
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0.0,0.0,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...,2.0,471.0,0.030492
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...,13.0,41.0,0.036130


Merge product features

In [10]:
article_features = (
    articles.merge(
        product_popularity,
        on='article_id',
        how='left'
    )
)

article_features.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc,purchase_count
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,7535.0
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,5666.0
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.,177.0
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",982.0
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde...",482.0


Fill missing engineered values

In [12]:
customer_features[
    'purchase_frequency'
] = customer_features[
    'purchase_frequency'
].fillna(0)

customer_features[
    'avg_spend'
] = customer_features[
    'avg_spend'
].fillna(0)

customer_features[
    'days_since_last_purchase'
] = customer_features[
    'days_since_last_purchase'
].fillna(999)

article_features[
    'purchase_count'
] = article_features[
    'purchase_count'
].fillna(0)

In [13]:
customer_features.to_parquet(
    "/kaggle/working/customer_features.parquet",
    index=False
)

article_features.to_parquet(
    "/kaggle/working/article_features.parquet",
    index=False
)

print("Feature engineering complete!")

Feature engineering complete!
